# Comparación principal de modelos para clasificación médica

Este es el notebook principal de experimentación del proyecto. Su objetivo es comparar, bajo un mismo protocolo, los modelos tradicionales y preentrenados para clasificar noticias médicas como:

- REFUTADA = la afirmación es incompatible con la evidencia médica revisada.
- RESPALDADA = la afirmación es compatible con la evidencia médica revisada.

La unidad experimental es una noticia identificada por record_id. El modelo recibe únicamente title + body. Claims, evidencia, PMIDs, fuente, tema y notas de revisión se conservan para auditoría, pero no se usan como variables predictoras.

El notebook no inventa resultados: si faltan datos, dependencias, embeddings o pesos de un modelo, registra el estado como blocked o skipped.

## Protocolo común

Todos los modelos se entrenan y evalúan con:

- La misma entrada binaria validada.
- La misma semilla (42).
- La misma partición estratificada 70 % train / 15 % validation / 15 % test.
- El mismo conjunto final de prueba, aislado de la selección.
- Las mismas métricas principales: Macro-F1 y F1 de REFUTADA.
- Métricas complementarias: accuracy, precision, recall, ROC-AUC y matriz de confusión.

Modelos contemplados:

1. Baseline de clase mayoritaria.
2. TF-IDF + Naive Bayes.
3. TF-IDF + Regresión Logística.
4. TF-IDF + SVM lineal.
5. Word2Vec + LSTM.
6. GloVe + BiLSTM.
7. BERT multilingüe.
8. BETO.
9. RoBERTa-BNE.
10. SBERT preentrenado + clasificador lineal.

Los modelos secuenciales y preentrenados son opcionales porque requieren pesos y dependencias que no deben descargarse automáticamente durante una ejecución reproducible.

In [ ]:
from __future__ import annotations

import copy
import json
import os
import random
import re
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    import numpy as np
    import pandas as pd
except ImportError as exc:
    raise RuntimeError(
        "Instala primero numpy y pandas. El notebook no instala paquetes automáticamente."
    ) from exc

try:
    from IPython.display import display
except ImportError:
    display = print


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "scripts" / "03_train_baseline.py").is_file()
        ):
            return candidate
    raise FileNotFoundError(
        "No se encontró la raíz del repositorio. Abre el notebook desde el checkout "
        "peruvian-medical-misinformation-classification."
    )


REPO_ROOT = find_repo_root()
EXPERIMENT_ROOT = REPO_ROOT / "experiments"
ARTIFACT_ROOT = REPO_ROOT / "artifacts" / "training_runs" / "main_model_comparison"

# Déjalo en None para buscar un training.csv válido dentro del repositorio.
# Si aparecen varios, especifica aquí el archivo exacto antes de entrenar.
INPUT_CSV: Path | None = None

SEED = 42
MIN_ROWS = 100
MIN_ROWS_PER_CLASS = 10
MAX_VOCAB_SIZE = 30_000
MAX_SEQUENCE_LENGTH = 300
MAX_TOKEN_LENGTH = 256

# El notebook registra todos los modelos. Cambia una categoría a False si
# quieres hacer una corrida parcial.
RUN_CLASSICAL = True
RUN_SEQUENCE_MODELS = True
RUN_TRANSFORMERS = True
RUN_SBERT = True

# No se descargan pesos por defecto. Para permitir una descarga explícita,
# cambia ALLOW_MODEL_DOWNLOAD a True y vuelve a ejecutar las celdas de configuración.
ALLOW_MODEL_DOWNLOAD = False
LOCAL_FILES_ONLY = not ALLOW_MODEL_DOWNLOAD

SAVE_MODELS = True
SEQUENCE_EPOCHS = 5
TRANSFORMER_EPOCHS = 3
TRANSFORMER_BATCH_SIZE = 8
TRANSFORMER_LEARNING_RATE = 2e-5
SBERT_BATCH_SIZE = 16

# Rutas opcionales a embeddings/modelos locales.
WORD2VEC_PATH: Path | None = None
GLOVE_PATH: Path | None = None
SBERT_MODEL_PATH: Path | None = None
SBERT_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

TRANSFORMER_MODELS = [
    {
        "key": "bert",
        "name": "BERT multilingüe",
        "model_name": "bert-base-multilingual-cased",
    },
    {
        "key": "beto",
        "name": "BETO",
        "model_name": "dccuchile/bert-base-spanish-wwm-cased",
    },
    {
        "key": "roberta_bne",
        "name": "RoBERTa-BNE",
        "model_name": "PlanTL-GOB-ES/roberta-base-bne",
    },
]

MODEL_CATALOG = [
    ("majority", "Mayoritaria", "baseline"),
    ("tfidf_naive_bayes", "TF-IDF + Naive Bayes", "classical"),
    ("tfidf_logistic_regression", "TF-IDF + Regresión Logística", "classical"),
    ("tfidf_linear_svm", "TF-IDF + SVM lineal", "classical"),
    ("word2vec_lstm", "Word2Vec + LSTM", "sequential"),
    ("glove_bilstm", "GloVe + BiLSTM", "sequential"),
    ("bert", "BERT multilingüe", "transformer"),
    ("beto", "BETO", "transformer"),
    ("roberta_bne", "RoBERTa-BNE", "transformer"),
    ("sbert", "SBERT + clasificador lineal", "sentence-embedding"),
]

LABEL_TO_ID = {"REFUTADA": 0, "RESPALDADA": 1}
ID_TO_LABEL = {value: key for key, value in LABEL_TO_ID.items()}
LABEL_ORDER = [0, 1]

random.seed(SEED)
np.random.seed(SEED)

print(f"Repositorio: {REPO_ROOT}")
print(f"Salida local ignorada por Git: {ARTIFACT_ROOT}")

## Dependencias

El notebook no instala paquetes ni descarga modelos por sí solo. El baseline tradicional necesita las dependencias de entrenamiento del proyecto. Los demás grupos son opcionales y pueden instalarse solo cuando corresponda.

En una terminal del entorno del repositorio:

    python3 -m pip install -e '.[training]'
    python3 -m pip install pandas numpy matplotlib
    # Opcionales para los modelos preentrenados:
    python3 -m pip install torch transformers sentence-transformers
    python3 -m pip install tensorflow gensim

Los pesos de BERT, BETO, RoBERTa-BNE y SBERT deben estar disponibles localmente o se debe habilitar explícitamente ALLOW_MODEL_DOWNLOAD.

In [ ]:
LABEL_COLUMNS_REQUIRED = {"record_id", "title", "body", "label"}


def build_text(row: Any) -> str:
    return f"{str(row['title']).strip()}\n\n{str(row['body']).strip()}"


def discover_training_csv() -> Path | None:
    if INPUT_CSV is not None:
        path = Path(INPUT_CSV).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"INPUT_CSV no existe: {path}")
        return path

    candidates: set[Path] = set()
    direct_candidates = [
        REPO_ROOT / "training.csv",
        REPO_ROOT / "labelstudio" / "artifacts" / "training.csv",
    ]
    for candidate in direct_candidates:
        if candidate.is_file():
            candidates.add(candidate.resolve())

    for search_root in (
        REPO_ROOT / "labelstudio" / "artifacts",
        REPO_ROOT / "artifacts",
    ):
        if search_root.is_dir():
            candidates.update(
                path.resolve()
                for path in search_root.rglob("training.csv")
                if path.is_file()
            )

    ordered = sorted(candidates, key=lambda path: str(path))
    if not ordered:
        return None
    if len(ordered) > 1:
        print("Se encontraron varios training.csv. Especifica INPUT_CSV:")
        for candidate in ordered:
            print(f"  - {candidate}")
        return None
    return ordered[0]


def load_binary_training_data(path: Path) -> tuple[pd.DataFrame, dict[str, Any]]:
    raw = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    missing_columns = sorted(LABEL_COLUMNS_REQUIRED.difference(raw.columns))
    if missing_columns:
        raise ValueError(
            f"Faltan columnas obligatorias: {', '.join(missing_columns)}"
        )

    raw = raw.copy()
    for column in raw.columns:
        raw[column] = raw[column].fillna("").astype(str).str.strip()

    if raw["record_id"].eq("").any():
        raise ValueError("Hay filas sin record_id.")
    duplicated = raw.loc[raw["record_id"].duplicated(keep=False), "record_id"].unique()
    if len(duplicated):
        raise ValueError(
            "Hay record_id duplicados; la unidad de entrenamiento debe ser una noticia única: "
            + ", ".join(map(str, duplicated[:10]))
        )

    total_rows = len(raw)
    binary = raw[raw["label"].str.upper().isin(LABEL_TO_ID)].copy()
    binary["label"] = binary["label"].str.upper()

    # Estos filtros mantienen la regla del baseline existente cuando el Gold
    # conserva filas de auditoría junto a las filas binarias.
    if "final_training_eligible" in binary.columns:
        eligible = binary["final_training_eligible"].str.upper().isin({"", "SI"})
        binary = binary[eligible].copy()
    if "validation_status" in binary.columns:
        validated = binary["validation_status"].str.upper().isin({"", "COMPLETADA"})
        binary = binary[validated].copy()

    missing_text = binary[
        binary["title"].eq("") | binary["body"].eq("")
    ]["record_id"].tolist()
    if missing_text:
        raise ValueError(
            "Hay filas binarias sin title o body: " + ", ".join(missing_text[:10])
        )

    binary = binary.reset_index(drop=True)
    if binary.empty:
        raise ValueError("El archivo no contiene filas binarias elegibles.")

    audit = {
        "rows_in_file": int(total_rows),
        "binary_rows": int(len(binary)),
        "excluded_non_binary": int(total_rows - len(binary)),
        "class_counts": {
            str(key): int(value)
            for key, value in binary["label"].value_counts().to_dict().items()
        },
    }
    return binary, audit


DATA_PATH = discover_training_csv()
DATA_DF: pd.DataFrame | None = None
DATA_AUDIT: dict[str, Any] = {}
DATA_BLOCKERS: list[str] = []

if DATA_PATH is None:
    DATA_BLOCKERS.append(
        "No se encontró un único training.csv. Define INPUT_CSV con la ruta del Gold binario."
    )
    print("BLOQUEADO: " + DATA_BLOCKERS[-1])
else:
    try:
        DATA_DF, DATA_AUDIT = load_binary_training_data(DATA_PATH)
        print(f"Entrada: {DATA_PATH}")
        print(f"Filas binarias elegibles: {len(DATA_DF)}")
        print(f"Clases: {DATA_AUDIT['class_counts']}")
    except Exception as exc:
        DATA_BLOCKERS.append(f"{type(exc).__name__}: {exc}")
        print("BLOQUEADO: " + DATA_BLOCKERS[-1])

In [ ]:
def show_frame(frame: pd.DataFrame) -> None:
    if frame.empty:
        print("(tabla vacía)")
    else:
        display(frame)


if DATA_DF is not None:
    summary = (
        DATA_DF["label"]
        .value_counts()
        .rename_axis("label")
        .reset_index(name="n")
    )
    summary["proportion"] = (summary["n"] / summary["n"].sum()).round(4)
    show_frame(summary)
else:
    print("No hay resumen porque la entrada todavía no está lista.")

## Partición estratificada y control de calidad

La columna dataset_split que pueda venir en el CSV se conserva como metadata, pero este notebook construye una partición determinista propia para que todos los modelos comparen exactamente los mismos artículos. El manifiesto de la partición se guarda junto con los resultados.

In [ ]:
def build_stratified_splits(dataframe: pd.DataFrame) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    from sklearn.model_selection import train_test_split

    if dataframe["label"].value_counts().min() < 4:
        raise ValueError(
            "Cada clase necesita al menos cuatro registros para separar train, validation y test."
        )

    indexes = np.arange(len(dataframe))
    labels = dataframe["label"].map(LABEL_TO_ID).to_numpy()

    train_indexes, remainder_indexes = train_test_split(
        indexes,
        test_size=0.30,
        random_state=SEED,
        stratify=labels,
    )
    remainder_labels = labels[remainder_indexes]
    validation_indexes, test_indexes = train_test_split(
        remainder_indexes,
        test_size=0.50,
        random_state=SEED,
        stratify=remainder_labels,
    )

    assignment = {}
    for index in train_indexes:
        assignment[int(index)] = "train"
    for index in validation_indexes:
        assignment[int(index)] = "validation"
    for index in test_indexes:
        assignment[int(index)] = "test"

    if len(assignment) != len(dataframe):
        raise RuntimeError("La partición no asignó exactamente una división a cada registro.")

    manifest = dataframe[
        [column for column in ("record_id", "label", "source_name", "topic") if column in dataframe]
    ].copy()
    manifest["dataset_split"] = [
        assignment[index] for index in range(len(dataframe))
    ]

    splits = {
        split_name: dataframe.loc[
            [index for index, assigned in assignment.items() if assigned == split_name]
        ].reset_index(drop=True)
        for split_name in ("train", "validation", "test")
    }
    return splits, manifest


SPLITS: dict[str, pd.DataFrame] = {}
SPLIT_MANIFEST: pd.DataFrame | None = None
TRAINING_READY = False

if DATA_DF is not None:
    counts = DATA_DF["label"].value_counts()
    if len(DATA_DF) < MIN_ROWS:
        DATA_BLOCKERS.append(
            f"Hay {len(DATA_DF)} filas binarias; el piloto exige al menos {MIN_ROWS}."
        )
    for label in LABEL_TO_ID:
        if int(counts.get(label, 0)) < MIN_ROWS_PER_CLASS:
            DATA_BLOCKERS.append(
                f"La clase {label} tiene {int(counts.get(label, 0))} filas; "
                f"se requieren al menos {MIN_ROWS_PER_CLASS}."
            )

    if not DATA_BLOCKERS:
        try:
            SPLITS, SPLIT_MANIFEST = build_stratified_splits(DATA_DF)
            TRAINING_READY = True
        except Exception as exc:
            DATA_BLOCKERS.append(f"{type(exc).__name__}: {exc}")

if TRAINING_READY:
    split_summary = pd.DataFrame(
        [
            {
                "split": split_name,
                "n": len(split_rows),
                "REFUTADA": int((split_rows["label"] == "REFUTADA").sum()),
                "RESPALDADA": int((split_rows["label"] == "RESPALDADA").sum()),
            }
            for split_name, split_rows in SPLITS.items()
        ]
    )
    show_frame(split_summary)
else:
    if DATA_BLOCKERS:
        print("No se ejecutará entrenamiento todavía:")
        for blocker in DATA_BLOCKERS:
            print(f"  - {blocker}")

## Métricas y registro común de resultados

Cada fila de comparison_metrics.csv representa un modelo en una división. La tabla final usa la división test. El valor de score_respaldada se interpreta como probabilidad o puntuación a favor de RESPALDADA, según el modelo.

In [ ]:
RESULT_ROWS: list[dict[str, Any]] = []
PREDICTION_ROWS: list[dict[str, Any]] = []
MODEL_LOGS: list[dict[str, Any]] = []


def add_skip(model_key: str, model_name: str, family: str, status: str, error: str) -> None:
    RESULT_ROWS.append(
        {
            "model_key": model_key,
            "model": model_name,
            "family": family,
            "split": "test",
            "status": status,
            "n": None,
            "accuracy": None,
            "macro_precision": None,
            "macro_recall": None,
            "macro_f1": None,
            "refutada_f1": None,
            "respaldada_f1": None,
            "roc_auc": None,
            "fit_seconds": None,
            "confusion_matrix": None,
            "error": error,
        }
    )
    MODEL_LOGS.append(
        {
            "model_key": model_key,
            "model": model_name,
            "family": family,
            "status": status,
            "error": error,
        }
    )


def evaluate_predictions(
    model_key: str,
    model_name: str,
    family: str,
    split_name: str,
    record_ids: list[str],
    y_true: Any,
    y_pred: Any,
    y_score: Any | None,
    fit_seconds: float,
) -> dict[str, Any]:
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
        roc_auc_score,
    )

    true_array = np.asarray(y_true, dtype=int)
    pred_array = np.asarray(y_pred, dtype=int)
    score_array = None if y_score is None else np.asarray(y_score, dtype=float)

    report = classification_report(
        true_array,
        pred_array,
        labels=LABEL_ORDER,
        target_names=[ID_TO_LABEL[index] for index in LABEL_ORDER],
        output_dict=True,
        zero_division=0,
    )
    matrix = confusion_matrix(true_array, pred_array, labels=LABEL_ORDER).tolist()

    row: dict[str, Any] = {
        "model_key": model_key,
        "model": model_name,
        "family": family,
        "split": split_name,
        "status": "ok",
        "n": int(len(true_array)),
        "accuracy": round(float(accuracy_score(true_array, pred_array)), 6),
        "macro_precision": round(
            float(precision_score(true_array, pred_array, average="macro", zero_division=0)), 6
        ),
        "macro_recall": round(
            float(recall_score(true_array, pred_array, average="macro", zero_division=0)), 6
        ),
        "macro_f1": round(
            float(f1_score(true_array, pred_array, average="macro", zero_division=0)), 6
        ),
        "refutada_f1": round(float(report["REFUTADA"]["f1-score"]), 6),
        "respaldada_f1": round(float(report["RESPALDADA"]["f1-score"]), 6),
        "roc_auc": None,
        "fit_seconds": round(float(fit_seconds), 3),
        "confusion_matrix": json.dumps(matrix, ensure_ascii=False),
        "error": "",
    }

    if score_array is not None and len(np.unique(true_array)) == 2:
        row["roc_auc"] = round(float(roc_auc_score(true_array, score_array)), 6)

    RESULT_ROWS.append(row)

    for record_id, true_id, predicted_id, score in zip(
        record_ids, true_array, pred_array, score_array if score_array is not None else [None] * len(true_array)
    ):
        PREDICTION_ROWS.append(
            {
                "model_key": model_key,
                "model": model_name,
                "dataset_split": split_name,
                "record_id": record_id,
                "true_label": ID_TO_LABEL[int(true_id)],
                "predicted_label": ID_TO_LABEL[int(predicted_id)],
                "correct": int(true_id == predicted_id),
                "score_respaldada": None if score is None else round(float(score), 6),
            }
        )

    return row


def model_texts(split_name: str) -> list[str]:
    return [build_text(row) for _, row in SPLITS[split_name].iterrows()]


def model_labels(split_name: str) -> np.ndarray:
    return SPLITS[split_name]["label"].map(LABEL_TO_ID).to_numpy(dtype=int)


def model_record_ids(split_name: str) -> list[str]:
    return SPLITS[split_name]["record_id"].tolist()

## Modelos tradicionales

Estos modelos forman la comparación base. TF-IDF se ajusta únicamente sobre train; la validación y la prueba solo se transforman con el vectorizador ya ajustado.

In [ ]:
def run_sklearn_model(
    model_key: str,
    model_name: str,
    family: str,
    estimator: Any,
) -> None:
    if not TRAINING_READY:
        add_skip(model_key, model_name, family, "blocked", "; ".join(DATA_BLOCKERS))
        return

    try:
        from joblib import dump
    except ImportError:
        dump = None

    try:
        started = time.perf_counter()
        estimator.fit(model_texts("train"), model_labels("train"))
        fit_seconds = time.perf_counter() - started

        for split_name in ("train", "validation", "test"):
            texts = model_texts(split_name)
            predictions = np.asarray(estimator.predict(texts), dtype=int)
            scores = None
            if hasattr(estimator, "predict_proba"):
                scores = np.asarray(estimator.predict_proba(texts))[:, 1]
            elif hasattr(estimator, "decision_function"):
                scores = np.asarray(estimator.decision_function(texts))

            evaluate_predictions(
                model_key=model_key,
                model_name=model_name,
                family=family,
                split_name=split_name,
                record_ids=model_record_ids(split_name),
                y_true=model_labels(split_name),
                y_pred=predictions,
                y_score=scores,
                fit_seconds=fit_seconds,
            )

        if SAVE_MODELS and dump is not None:
            model_dir = ARTIFACT_ROOT / model_key
            model_dir.mkdir(parents=True, exist_ok=True)
            dump(estimator, model_dir / "model.joblib")

        MODEL_LOGS.append(
            {
                "model_key": model_key,
                "model": model_name,
                "family": family,
                "status": "ok",
                "fit_seconds": round(float(fit_seconds), 3),
                "error": "",
            }
        )
    except Exception as exc:
        add_skip(
            model_key,
            model_name,
            family,
            "error",
            f"{type(exc).__name__}: {exc}",
        )


def run_classical_models() -> None:
    if not TRAINING_READY:
        for model_key, model_name, family in MODEL_CATALOG[:4]:
            add_skip(model_key, model_name, family, "blocked", "; ".join(DATA_BLOCKERS))
        return

    try:
        from sklearn.dummy import DummyClassifier
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.linear_model import LogisticRegression
        from sklearn.naive_bayes import MultinomialNB
        from sklearn.pipeline import Pipeline
        from sklearn.svm import LinearSVC
    except ImportError:
        message = (
            "Faltan scikit-learn/joblib. Ejecuta: "
            "python3 -m pip install -e '.[training]'"
        )
        for model_key, model_name, family in MODEL_CATALOG[:4]:
            add_skip(model_key, model_name, family, "skipped", message)
        return

    vectorizer_config = {
        "lowercase": True,
        "strip_accents": "unicode",
        "ngram_range": (1, 2),
        "min_df": 1,
        "sublinear_tf": True,
        "max_features": 30_000,
    }

    models = [
        (
            "majority",
            "Mayoritaria",
            "baseline",
            DummyClassifier(strategy="most_frequent"),
        ),
        (
            "tfidf_naive_bayes",
            "TF-IDF + Naive Bayes",
            "classical",
            Pipeline(
                [
                    ("tfidf", TfidfVectorizer(**vectorizer_config)),
                    ("classifier", MultinomialNB(alpha=0.5)),
                ]
            ),
        ),
        (
            "tfidf_logistic_regression",
            "TF-IDF + Regresión Logística",
            "classical",
            Pipeline(
                [
                    ("tfidf", TfidfVectorizer(**vectorizer_config)),
                    (
                        "classifier",
                        LogisticRegression(
                            class_weight="balanced",
                            max_iter=1_000,
                            random_state=SEED,
                            solver="liblinear",
                        ),
                    ),
                ]
            ),
        ),
        (
            "tfidf_linear_svm",
            "TF-IDF + SVM lineal",
            "classical",
            Pipeline(
                [
                    ("tfidf", TfidfVectorizer(**vectorizer_config)),
                    (
                        "classifier",
                        LinearSVC(
                            class_weight="balanced",
                            random_state=SEED,
                        ),
                    ),
                ]
            ),
        ),
    ]

    for model_key, model_name, family, estimator in models:
        run_sklearn_model(model_key, model_name, family, estimator)


if RUN_CLASSICAL:
    run_classical_models()
else:
    print("Modelos tradicionales desactivados por configuración.")

## Word2Vec + LSTM y GloVe + BiLSTM

Los embeddings estáticos se leen desde rutas locales configuradas arriba. El tokenizer se ajusta solo sobre el texto de entrenamiento. Si no se proporciona el archivo de embeddings, el modelo queda registrado como skipped y no se sustituye silenciosamente por vectores aleatorios.

In [ ]:
def resolve_existing_path(path_value: Path | None) -> Path | None:
    if path_value is None:
        return None
    path = Path(path_value).expanduser().resolve()
    return path if path.is_file() else None


def embedding_matrix_from_word2vec(tokenizer: Any, path: Path) -> tuple[np.ndarray, int]:
    from gensim.models import KeyedVectors

    binary = str(path).lower().endswith((".bin", ".bin.gz"))
    vectors = KeyedVectors.load_word2vec_format(str(path), binary=binary)
    dimension = int(vectors.vector_size)
    vocabulary_size = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)
    matrix = np.zeros((vocabulary_size, dimension), dtype=np.float32)

    for token, index in tokenizer.word_index.items():
        if index >= vocabulary_size:
            continue
        try:
            matrix[index] = np.asarray(vectors[token], dtype=np.float32)
        except KeyError:
            continue
    return matrix, dimension


def embedding_matrix_from_glove(tokenizer: Any, path: Path) -> tuple[np.ndarray, int]:
    vocabulary_size = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)
    dimension: int | None = None
    rows: list[tuple[int, np.ndarray]] = []

    with path.open("r", encoding="utf-8", errors="ignore") as handle:
        for line_number, line in enumerate(handle, start=1):
            parts = line.strip().split()
            if not parts:
                continue
            if (
                line_number == 1
                and len(parts) == 2
                and all(part.isdigit() for part in parts)
            ):
                continue

            word = parts[0]
            values = parts[1:]
            if dimension is None:
                dimension = len(values)
            if len(values) != dimension:
                continue

            token_index = tokenizer.word_index.get(word)
            if token_index is None or token_index >= vocabulary_size:
                continue
            try:
                vector = np.asarray(values, dtype=np.float32)
            except ValueError:
                continue
            rows.append((token_index, vector))

    if dimension is None:
        raise ValueError(f"No se pudo inferir la dimensión de GloVe: {path}")

    matrix = np.zeros((vocabulary_size, dimension), dtype=np.float32)
    for token_index, vector in rows:
        matrix[token_index] = vector
    return matrix, dimension


def run_sequence_model(
    model_key: str,
    model_name: str,
    source_type: str,
    embedding_path_value: Path | None,
    bidirectional: bool,
) -> None:
    family = "sequential"

    if not TRAINING_READY:
        add_skip(model_key, model_name, family, "blocked", "; ".join(DATA_BLOCKERS))
        return

    embedding_path = resolve_existing_path(embedding_path_value)
    if embedding_path is None:
        add_skip(
            model_key,
            model_name,
            family,
            "skipped",
            "Configura una ruta local válida en WORD2VEC_PATH o GLOVE_PATH.",
        )
        return

    try:
        import tensorflow as tf
        from tensorflow.keras.callbacks import EarlyStopping
        from tensorflow.keras.layers import (
            LSTM,
            Bidirectional,
            Dense,
            Dropout,
            Embedding,
            Input,
        )
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.preprocessing.sequence import pad_sequences
        from tensorflow.keras.preprocessing.text import Tokenizer
    except ImportError:
        add_skip(
            model_key,
            model_name,
            family,
            "skipped",
            "Falta TensorFlow. Instala tensorflow para ejecutar este modelo.",
        )
        return

    try:
        tokenizer = Tokenizer(
            num_words=MAX_VOCAB_SIZE,
            oov_token="<OOV>",
            filters="",
        )
        tokenizer.fit_on_texts(model_texts("train"))

        train_sequences = tokenizer.texts_to_sequences(model_texts("train"))
        validation_sequences = tokenizer.texts_to_sequences(model_texts("validation"))
        test_sequences = tokenizer.texts_to_sequences(model_texts("test"))

        x_train = pad_sequences(
            train_sequences,
            maxlen=MAX_SEQUENCE_LENGTH,
            padding="post",
            truncating="post",
        )
        x_validation = pad_sequences(
            validation_sequences,
            maxlen=MAX_SEQUENCE_LENGTH,
            padding="post",
            truncating="post",
        )
        x_test = pad_sequences(
            test_sequences,
            maxlen=MAX_SEQUENCE_LENGTH,
            padding="post",
            truncating="post",
        )

        if source_type == "word2vec":
            embedding_matrix, embedding_dimension = embedding_matrix_from_word2vec(
                tokenizer, embedding_path
            )
        else:
            embedding_matrix, embedding_dimension = embedding_matrix_from_glove(
                tokenizer, embedding_path
            )

        model = Sequential(
            [
                Input(shape=(MAX_SEQUENCE_LENGTH,)),
                Embedding(
                    input_dim=embedding_matrix.shape[0],
                    output_dim=embedding_dimension,
                    weights=[embedding_matrix],
                    trainable=False,
                    mask_zero=True,
                ),
                (
                    Bidirectional(LSTM(64))
                    if bidirectional
                    else LSTM(64)
                ),
                Dropout(0.30),
                Dense(1, activation="sigmoid"),
            ]
        )
        model.compile(
            optimizer="adam",
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )

        started = time.perf_counter()
        model.fit(
            x_train,
            model_labels("train"),
            validation_data=(x_validation, model_labels("validation")),
            epochs=SEQUENCE_EPOCHS,
            batch_size=32,
            callbacks=[
                EarlyStopping(
                    monitor="val_loss",
                    patience=2,
                    restore_best_weights=True,
                )
            ],
            verbose=0,
        )
        fit_seconds = time.perf_counter() - started

        arrays = {
            "train": x_train,
            "validation": x_validation,
            "test": x_test,
        }
        for split_name, features in arrays.items():
            scores = model.predict(features, verbose=0).reshape(-1)
            predictions = (scores >= 0.5).astype(int)
            evaluate_predictions(
                model_key=model_key,
                model_name=model_name,
                family=family,
                split_name=split_name,
                record_ids=model_record_ids(split_name),
                y_true=model_labels(split_name),
                y_pred=predictions,
                y_score=scores,
                fit_seconds=fit_seconds,
            )

        if SAVE_MODELS:
            model_dir = ARTIFACT_ROOT / model_key
            model_dir.mkdir(parents=True, exist_ok=True)
            model.save(model_dir / "model.keras")

        MODEL_LOGS.append(
            {
                "model_key": model_key,
                "model": model_name,
                "family": family,
                "status": "ok",
                "fit_seconds": round(float(fit_seconds), 3),
                "embedding_path": str(embedding_path),
                "error": "",
            }
        )
        tf.keras.backend.clear_session()
    except Exception as exc:
        add_skip(
            model_key,
            model_name,
            family,
            "error",
            f"{type(exc).__name__}: {exc}",
        )


if RUN_SEQUENCE_MODELS:
    run_sequence_model(
        "word2vec_lstm",
        "Word2Vec + LSTM",
        "word2vec",
        WORD2VEC_PATH,
        bidirectional=False,
    )
    run_sequence_model(
        "glove_bilstm",
        "GloVe + BiLSTM",
        "glove",
        GLOVE_PATH,
        bidirectional=True,
    )
else:
    print("Modelos secuenciales desactivados por configuración.")

## BERT, BETO y RoBERTa-BNE

Estos modelos se ajustan como clasificadores binarios con una cabeza de clasificación. El código comparte tokenización, semilla, particiones y métricas, pero cada modelo conserva su propio peso preentrenado y su propia salida.

In [ ]:
class EncodedTextDataset:
    def __init__(self, encodings: dict[str, Any], labels: Any, torch_module: Any):
        self.encodings = encodings
        self.labels = torch_module.tensor(labels, dtype=torch_module.long)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> dict[str, Any]:
        item = {
            key: value[index]
            for key, value in self.encodings.items()
        }
        item["labels"] = self.labels[index]
        return item


def encode_transformer_texts(tokenizer: Any, texts: list[str]) -> dict[str, Any]:
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=MAX_TOKEN_LENGTH,
        return_tensors="pt",
    )


def transformer_predictions(
    model: Any,
    dataset: Any,
    torch_module: Any,
    device: Any,
) -> tuple[np.ndarray, np.ndarray]:
    from torch.utils.data import DataLoader

    loader = DataLoader(dataset, batch_size=TRANSFORMER_BATCH_SIZE, shuffle=False)
    predictions: list[np.ndarray] = []
    scores: list[np.ndarray] = []
    model.eval()

    with torch_module.no_grad():
        for batch in loader:
            labels = batch.pop("labels")
            del labels
            inputs = {
                key: value.to(device)
                for key, value in batch.items()
            }
            logits = model(**inputs).logits
            probabilities = torch_module.softmax(logits, dim=-1)[:, 1]
            scores.append(probabilities.detach().cpu().numpy())
            predictions.append(
                (probabilities >= 0.5).to(torch_module.int64).cpu().numpy()
            )

    return np.concatenate(predictions), np.concatenate(scores)


def run_transformer_model(model_spec: dict[str, str]) -> None:
    model_key = model_spec["key"]
    model_name = model_spec["name"]
    family = "transformer"

    if not TRAINING_READY:
        add_skip(model_key, model_name, family, "blocked", "; ".join(DATA_BLOCKERS))
        return

    try:
        import torch
        from torch.utils.data import DataLoader
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
    except ImportError:
        add_skip(
            model_key,
            model_name,
            family,
            "skipped",
            "Faltan torch y/o transformers para este modelo.",
        )
        return

    try:
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        tokenizer = AutoTokenizer.from_pretrained(
            model_spec["model_name"],
            local_files_only=LOCAL_FILES_ONLY,
        )
        model = AutoModelForSequenceClassification.from_pretrained(
            model_spec["model_name"],
            num_labels=2,
            id2label=ID_TO_LABEL,
            label2id=LABEL_TO_ID,
            local_files_only=LOCAL_FILES_ONLY,
        ).to(device)

        encoded = {
            split_name: encode_transformer_texts(
                tokenizer, model_texts(split_name)
            )
            for split_name in ("train", "validation", "test")
        }
        datasets = {
            split_name: EncodedTextDataset(
                encoded[split_name],
                model_labels(split_name),
                torch,
            )
            for split_name in ("train", "validation", "test")
        }
        train_loader = DataLoader(
            datasets["train"],
            batch_size=TRANSFORMER_BATCH_SIZE,
            shuffle=True,
        )
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=TRANSFORMER_LEARNING_RATE,
        )

        best_validation_f1 = -1.0
        best_state: dict[str, Any] | None = None
        epochs_without_improvement = 0
        started = time.perf_counter()

        for epoch in range(TRANSFORMER_EPOCHS):
            model.train()
            for batch in train_loader:
                labels = batch.pop("labels").to(device)
                inputs = {
                    key: value.to(device)
                    for key, value in batch.items()
                }
                optimizer.zero_grad()
                output = model(**inputs, labels=labels)
                output.loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            validation_predictions, _ = transformer_predictions(
                model, datasets["validation"], torch, device
            )
            from sklearn.metrics import f1_score

            validation_f1 = float(
                f1_score(
                    model_labels("validation"),
                    validation_predictions,
                    average="macro",
                    zero_division=0,
                )
            )
            print(
                f"{model_name}: época {epoch + 1}/{TRANSFORMER_EPOCHS}, "
                f"validation Macro-F1={validation_f1:.4f}"
            )

            if validation_f1 > best_validation_f1:
                best_validation_f1 = validation_f1
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if epochs_without_improvement >= 2:
                    break

        fit_seconds = time.perf_counter() - started
        if best_state is not None:
            model.load_state_dict(best_state)

        for split_name in ("train", "validation", "test"):
            predictions, scores = transformer_predictions(
                model, datasets[split_name], torch, device
            )
            evaluate_predictions(
                model_key=model_key,
                model_name=model_name,
                family=family,
                split_name=split_name,
                record_ids=model_record_ids(split_name),
                y_true=model_labels(split_name),
                y_pred=predictions,
                y_score=scores,
                fit_seconds=fit_seconds,
            )

        if SAVE_MODELS:
            model_dir = ARTIFACT_ROOT / model_key
            model_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(model_dir)
            tokenizer.save_pretrained(model_dir)

        MODEL_LOGS.append(
            {
                "model_key": model_key,
                "model": model_name,
                "family": family,
                "status": "ok",
                "fit_seconds": round(float(fit_seconds), 3),
                "device": str(device),
                "model_name": model_spec["model_name"],
                "error": "",
            }
        )
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as exc:
        add_skip(
            model_key,
            model_name,
            family,
            "error",
            f"{type(exc).__name__}: {exc}",
        )


if RUN_TRANSFORMERS:
    for transformer_spec in TRANSFORMER_MODELS:
        run_transformer_model(transformer_spec)
else:
    print("Transformers desactivados por configuración.")

## SBERT + clasificador lineal

SBERT se usa aquí como encoder de frases preentrenado y se evalúa con un clasificador lineal. La similitud semántica no se interpreta como verdad médica: el objetivo sigue siendo la clasificación supervisada usando únicamente las etiquetas humanas del Gold.

In [ ]:
def run_sbert_model() -> None:
    model_key = "sbert"
    model_name = "SBERT + clasificador lineal"
    family = "sentence-embedding"

    if not TRAINING_READY:
        add_skip(model_key, model_name, family, "blocked", "; ".join(DATA_BLOCKERS))
        return

    if SBERT_MODEL_PATH is None and not ALLOW_MODEL_DOWNLOAD:
        add_skip(
            model_key,
            model_name,
            family,
            "skipped",
            "Configura SBERT_MODEL_PATH o habilita ALLOW_MODEL_DOWNLOAD explícitamente.",
        )
        return

    try:
        from joblib import dump
        from sentence_transformers import SentenceTransformer
        from sklearn.linear_model import LogisticRegression
    except ImportError:
        add_skip(
            model_key,
            model_name,
            family,
            "skipped",
            "Faltan sentence-transformers, scikit-learn o joblib.",
        )
        return

    try:
        source = (
            str(SBERT_MODEL_PATH.expanduser().resolve())
            if SBERT_MODEL_PATH is not None
            else SBERT_MODEL_NAME
        )
        device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
        model_kwargs = {"local_files_only": LOCAL_FILES_ONLY}

        try:
            encoder = SentenceTransformer(
                source,
                device=device,
                model_kwargs=model_kwargs,
            )
        except TypeError:
            if LOCAL_FILES_ONLY:
                raise
            encoder = SentenceTransformer(source, device=device)

        embeddings: dict[str, np.ndarray] = {}
        for split_name in ("train", "validation", "test"):
            embeddings[split_name] = np.asarray(
                encoder.encode(
                    model_texts(split_name),
                    batch_size=SBERT_BATCH_SIZE,
                    show_progress_bar=True,
                    normalize_embeddings=True,
                )
            )

        classifier = LogisticRegression(
            class_weight="balanced",
            max_iter=1_000,
            random_state=SEED,
            solver="liblinear",
        )
        started = time.perf_counter()
        classifier.fit(embeddings["train"], model_labels("train"))
        fit_seconds = time.perf_counter() - started

        for split_name in ("train", "validation", "test"):
            predictions = classifier.predict(embeddings[split_name])
            scores = classifier.predict_proba(embeddings[split_name])[:, 1]
            evaluate_predictions(
                model_key=model_key,
                model_name=model_name,
                family=family,
                split_name=split_name,
                record_ids=model_record_ids(split_name),
                y_true=model_labels(split_name),
                y_pred=predictions,
                y_score=scores,
                fit_seconds=fit_seconds,
            )

        if SAVE_MODELS and dump is not None:
            model_dir = ARTIFACT_ROOT / model_key
            model_dir.mkdir(parents=True, exist_ok=True)
            dump(classifier, model_dir / "classifier.joblib")
            (model_dir / "encoder_config.json").write_text(
                json.dumps(
                    {
                        "encoder": source,
                        "normalize_embeddings": True,
                    },
                    ensure_ascii=False,
                    indent=2,
                ),
                encoding="utf-8",
            )

        MODEL_LOGS.append(
            {
                "model_key": model_key,
                "model": model_name,
                "family": family,
                "status": "ok",
                "fit_seconds": round(float(fit_seconds), 3),
                "encoder": source,
                "error": "",
            }
        )
    except Exception as exc:
        add_skip(
            model_key,
            model_name,
            family,
            "error",
            f"{type(exc).__name__}: {exc}",
        )


if RUN_SBERT:
    run_sbert_model()
else:
    print("SBERT desactivado por configuración.")

## Ejecución

La siguiente celda ejecuta todos los grupos activados arriba. Si el dataset todavía no cumple los controles de calidad, se conserva una fila blocked por modelo esperado para que la tabla explique por qué no hay resultados.

In [ ]:
if not TRAINING_READY:
    for model_key, model_name, family in MODEL_CATALOG:
        if not any(row["model_key"] == model_key for row in RESULT_ROWS):
            add_skip(model_key, model_name, family, "blocked", "; ".join(DATA_BLOCKERS))
else:
    print("Ejecución completada para las categorías activadas.")

## Tabla comparativa y artefactos

La tabla que debe usarse para la comparación principal es la correspondiente a test. Las filas skipped, blocked y error no se convierten en ceros: se conservan como estados explícitos.

In [ ]:
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

RESULTS_DF = pd.DataFrame(RESULT_ROWS)
PREDICTIONS_DF = pd.DataFrame(PREDICTION_ROWS)
LOGS_DF = pd.DataFrame(MODEL_LOGS)

if SPLIT_MANIFEST is not None:
    SPLIT_MANIFEST.to_csv(
        ARTIFACT_ROOT / "split_manifest.csv",
        index=False,
        encoding="utf-8-sig",
    )

RESULTS_DF.to_csv(
    ARTIFACT_ROOT / "comparison_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)
PREDICTIONS_DF.to_csv(
    ARTIFACT_ROOT / "predictions.csv",
    index=False,
    encoding="utf-8-sig",
)
LOGS_DF.to_csv(
    ARTIFACT_ROOT / "model_run_log.csv",
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "input_csv": None if DATA_PATH is None else str(DATA_PATH),
    "seed": SEED,
    "split": "70/15/15 stratified",
    "features": ["title", "body"],
    "target": "label",
    "label_mapping": LABEL_TO_ID,
    "data_audit": DATA_AUDIT,
    "data_blockers": DATA_BLOCKERS,
    "config": {
        "run_classical": RUN_CLASSICAL,
        "run_sequence_models": RUN_SEQUENCE_MODELS,
        "run_transformers": RUN_TRANSFORMERS,
        "run_sbert": RUN_SBERT,
        "allow_model_download": ALLOW_MODEL_DOWNLOAD,
        "local_files_only": LOCAL_FILES_ONLY,
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "max_token_length": MAX_TOKEN_LENGTH,
    },
}
(ARTIFACT_ROOT / "run_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

if RESULTS_DF.empty:
    print("No hay filas de resultados.")
else:
    test_results = RESULTS_DF[RESULTS_DF["split"] == "test"].copy()
    if not test_results.empty:
        test_results["_sort_macro_f1"] = pd.to_numeric(
            test_results["macro_f1"], errors="coerce"
        )
        test_results = test_results.sort_values(
            ["status", "_sort_macro_f1"],
            ascending=[True, False],
            na_position="last",
        ).drop(columns="_sort_macro_f1")

        table_columns = [
            "model",
            "family",
            "status",
            "n",
            "accuracy",
            "macro_precision",
            "macro_recall",
            "macro_f1",
            "refutada_f1",
            "respaldada_f1",
            "roc_auc",
            "fit_seconds",
            "error",
        ]
        show_frame(test_results[table_columns])
    else:
        print("No hay resultados de test.")

## Visualización rápida

La figura solo se genera para modelos que realmente terminaron con estado ok. No se grafican estados bloqueados como si fueran métricas de cero.

In [ ]:
try:
    import matplotlib.pyplot as plt

    if RESULTS_DF.empty:
        print("No hay datos para graficar.")
    else:
        plot_data = RESULTS_DF[
            (RESULTS_DF["split"] == "test")
            & (RESULTS_DF["status"] == "ok")
        ].copy()
        plot_data["macro_f1"] = pd.to_numeric(plot_data["macro_f1"], errors="coerce")
        plot_data = plot_data.dropna(subset=["macro_f1"]).sort_values("macro_f1")

        if plot_data.empty:
            print("Todavía no hay modelos ejecutados correctamente.")
        else:
            figure, axis = plt.subplots(figsize=(10, 5))
            axis.barh(plot_data["model"], plot_data["macro_f1"], color="#2563eb")
            axis.set_xlim(0, 1)
            axis.set_xlabel("Macro-F1 en test")
            axis.set_title("Comparación principal de modelos")
            axis.grid(axis="x", alpha=0.25)
            figure.tight_layout()
            figure.savefig(
                ARTIFACT_ROOT / "comparison_macro_f1.png",
                dpi=160,
                bbox_inches="tight",
            )
            plt.show()
except ImportError:
    print("matplotlib no está instalado; se conserva la tabla CSV.")

## Lectura de resultados

Para la discusión de la tesis:

1. Ordena primero por Macro-F1.
2. Revisa F1 de REFUTADA, porque es la clase de interés para detectar desinformación.
3. Comprueba la matriz de confusión y no uses Accuracy como único argumento.
4. Reporta el tamaño y la distribución del test.
5. Describe qué modelos no se ejecutaron y por qué.
6. No compares directamente estas métricas de clasificación con Precision@k, MRR o nDCG; esas métricas pertenecen a una tarea distinta de recuperación de evidencia.

Los archivos generados quedan en:

- artifacts/training_runs/main_model_comparison/comparison_metrics.csv
- artifacts/training_runs/main_model_comparison/predictions.csv
- artifacts/training_runs/main_model_comparison/split_manifest.csv
- artifacts/training_runs/main_model_comparison/run_metadata.json
- artifacts/training_runs/main_model_comparison/comparison_macro_f1.png

Los artefactos locales están excluidos de Git. El notebook sí conserva el protocolo y el código reproducible.